# 學生學習與作業版｜任務 10：預測時間與資料洩漏

**姓名：**歐國澤 **學號／組別：**C114118111 **完成日期：**2026-11-26

**學習目標：** 辨認預測時不可用的欄位，區分標籤、特徵與事後資訊。

本單元與教師版「任務 10」相同標題、資料、教學順序與範例；除了明確標示的作業與填空，其餘教師版教材內容保留。對照重點：程式填空對應教師版「10.3 直接看答案」。

### 如何完成
1. 先閱讀觀念與執行資料準備；各單元可以獨立啟動，不需要依賴上一單元的變數。
2. **💻 P 作業**：在指定程式 cell 補上 `____`，可先用提供的 Vibe Coding 提示詞詢問 AI。補完再執行，並執行後面的驗證 cell。未填時出現 `NameError` 是預期現象，先完成填空再繼續。
3. **📝 Q 作業**：查詢 AI、比較教材內容，填寫緊接的文字作答區；請記錄實際提示詞、自己的解釋與查證方式。
4. **🎯 M 小任務**：在任務 4、8、12、16 將前面幾單元整合成可執行的成果。不要只交 AI 回覆，要交實際程式、輸出與自己的解釋。
5. 把 Notebook 存到自己的 Google 雲端硬碟，完成後下載／依教師指示繳交 `.ipynb`；小任務保留輸出、文字與 AI 使用紀錄。

**案例限制：**本資料是公式生成的 60 筆模擬紀錄，不能推論真實學生或因果。缺失與 15 小時值在計分後加入。預測時欄位可得性是假設；全部資料已在前半段探索，後續切出的測試集僅示範流程，不能視為完全獨立的新資料。

**完成檢核：**本單元有 1 項程式填空、2 項 AI 查詢作答；若是任務 4、8、12、16，另外有 1 項整合小任務。請保留自己的輸出、回答和驗證紀錄。


## 共用資料準備
以下在記憶體重建與既有教材相同的資料。執行後可以直接開始本單元；不會覆寫 CSV。


In [1]:
# 匯入套件：numpy 建立隨機資料，pandas 管理表格，pyplot 繪圖
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 固定種子，並維持呼叫順序，以重現原教材的 60 筆模擬資料
rng = np.random.default_rng(42)
n = 60
departments = ["資訊管理", "企業管理", "金融管理"]
# 每個字典鍵是一個欄位；對應的值是 60 名學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)
# 成績依設定公式與隨機誤差建立；關係來自模擬規則，不能視為因果發現
noise = rng.normal(0, 5, n)
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)
# 故意在計分後加入資料品質問題；無法從表格還原計分時原始值
df.loc[[5, 17], "sleep_hours"] = np.nan
df.loc[8, "screen_time_hours"] = 15.0
print("模擬資料形狀（列, 欄）：", df.shape)


模擬資料形狀（列, 欄）： (60, 7)


## 10.1 欄位可用性
**觀念說明：**在考前預測，考後才知道的 `quiz_score` 不可作輸入；`needs_support` 本身是標籤，也不能當特徵。`student_id` 是識別碼通常排除。其他行為欄位的考前可得性僅為教學假設。


In [2]:
# 建立教師可投影的欄位檢查表
availability = pd.DataFrame({
 "field": ["student_id", "department", "study_hours", "sleep_hours", "screen_time_hours", "attendance_rate", "quiz_score", "needs_support"],
 "decision": ["排除", "假設可用", "假設可用", "假設可用", "假設可用", "假設可用", "排除", "標籤，不可作輸入"],
 "reason": ["識別碼", "需確認可得時間", "需確認可得時間", "需確認可得時間", "需確認可得時間", "需確認可得時間", "考後才知道，且用來建標籤", "答案本身"]})
display(availability)


,field,decision,reason
0,student_id,排除,識別碼
1,department,假設可用,需確認可得時間
2,study_hours,假設可用,需確認可得時間
3,sleep_hours,假設可用,需確認可得時間
4,screen_time_hours,假設可用,需確認可得時間
5,attendance_rate,假設可用,需確認可得時間
6,quiz_score,排除,考後才知道，且用來建標籤
7,needs_support,標籤，不可作輸入,答案本身


## 📝 學生作業 10-Q1｜AI 查詢與作答：退訂洩漏
**任務：**預測下個月退訂時，為何退訂後才有的取消日期不可作輸入？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請用時間線說明客戶取消日期造成目標洩漏，並舉例可考慮的預測前欄位。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 10-Q1 專用作答區：請雙擊此文字儲存格填寫
- AI 工具及實際提示詞：Gemini；使用題目提供的提示詞詢問為何預測下個月退訂時，退訂後才有的取消日期不可作輸入，並請舉例預測前可用欄位。
- AI 回答重點（請自行整理）：若在月底（預測時間點）要預測下個月是否退訂，「取消日期（`cancellation_date`）」只有在客戶真的發生退訂行為後才會被系統寫入，因此它直接包含了未來的答案資訊。如果把它放進模型訓練，會造成嚴重的「目標洩漏（Target Leakage）」，讓模型在歷史資料上看似準確率極高，但實際上線預測時根本拿不到未來的取消日期而完全失效。
- 我的理解與具體例子：就像 10.2 節的小表 `churn`，我們在 1 月 31 日預測客戶 2 月是否退訂，客戶 A 與 C 是在 2 月 5 日和 2 月 12 日才取消訂閱（`cancellation_date` 有日期），B 沒退訂所以是 `None`。這代表在 1 月 31 日當下，所有人的取消日期都還是空的，根本不可能拿 2 月才發生的日期當輸入；合法的候選特徵應該是截止於 1 月 31 日前就能算出的行為指標，例如 `logins_before_cutoff`（1 月份登入次數）或過去 30 天客服申訴次數。
- 查證來源／方式及日期（無法查證請註明）：對照本單元教材 10.2 節 `churn` 測試表（`logins_before_cutoff` 與 `cancellation_date`）核對，日期：2026-11-27。
- 我檢查或修正 AI 回答的地方：直接套用 10.2 節 `churn` 表格裡的客戶 A、B、C 與 `2026-02-05`、`2026-02-12` 具體日期來說明時間線順序。


## 10.2 退訂案例
如果在本月底預測下個月是否退訂，且取消日期只有退訂後才產生，`cancellation_date` 就不能作為本月底預測的輸入；最近登入等欄位也必須確認截點與統計視窗。


In [3]:
# 建立獨立小表，對照預測當下能否使用
churn = pd.DataFrame({"customer_id": ["A", "B", "C"],
 "logins_before_cutoff": [3, 10, 1], "cancellation_date": ["2026-02-05", None, "2026-02-12"],
 "churn_next_month": [1, 0, 1]})
display(churn)
print("可以考慮的候選：", ["logins_before_cutoff"])
print("排除的事後欄位：", ["cancellation_date"])


,customer_id,logins_before_cutoff,cancellation_date,churn_next_month
0,A,3,2026-02-05,1
1,B,10,None,0
2,C,1,2026-02-12,1


可以考慮的候選： ['logins_before_cutoff']
排除的事後欄位： ['cancellation_date']


## 📝 學生作業 10-Q2｜AI 查詢與作答：欄位時間
**任務：**看見欄位名稱後，還要確認哪些收集或計算時間，才能判斷是否可用？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請說明截點、統計窗口、欄位可得時間如何影響預測特徵是否合法使用。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 10-Q2 專用作答區：請雙擊此文字儲存格填寫
- AI 工具及實際提示詞：Gemini；使用題目提供的提示詞詢問看到欄位名稱後，還要確認哪些截點、統計窗口與可得時間才能判斷是否合法使用。
- AI 回答重點（請自行整理）：不能只看欄位名稱看起來像事前行為就直接放進模型，必須確認三個時間條件：1. **預測截點（Cutoff Time）**：什麼時間點要做預測；2. **統計窗口（Aggregation Window）**：這個數值是從哪一天到哪一天加總或平均出來的，結尾有沒有超過預測截點；3. **欄位可得時間（Availability / Latency）**：資料表裡的數值是否會在事後被覆寫更新（例如學期末才結算整學期的總出席率，而不是考前當下的累計出席率）。
- 我的理解與具體例子：例如在學生資料表中看到 `attendance_rate`（出席率），雖然名字聽起來是平時表現，但如果這張資料表是在「期末考結束後」才統一匯出，裡面的 `attendance_rate` 其實包含了「考後後半學期」的點名紀錄，那在期中考前做預測時就會發生時間穿越的資料洩漏；因此必須確認計算窗口截止於「考前一天 23:59」且系統在考前確實已入帳。
- 查證來源／方式及日期（無法查證請註明）：對照本單元教材 10.1 節 `availability` 檢查表（「假設可用，需確認可得時間」）與延伸節說明核對，日期：2026-11-27。
- 我檢查或修正 AI 回答的地方：結合 10.1 節的 `attendance_rate` 與期末事後覆寫情境，讓三個時間概念更容易在真實系統中核對。


## 10.3 直接看答案的危險示範
下列程式不是合格模型：它直接使用事後分數計算答案。高正確率只能證明標籤規則被重複，不能證明可以在考前預測。


## 💻 學生作業 10-P1｜Vibe Coding 程式填空
**對應教師版：**任務 10「10.3 直接看答案」的示範程式。

**任務：**刻意利用考後 quiz_score，依低於 75 產生 leaked_answer，比較與 actual；這是錯誤情境示範，不是可部署模型。

**執行方式：**找到下一個程式儲存格中的 `____`，使用自己撰寫或 AI 建議的程式補齊，保留其他已提供的程式與註解。補完後先執行該 cell，再執行緊接的自動檢查；後續單元內容可能依賴此變數。

**可使用的 Vibe Coding 提示詞：**
```text
我是 Python 與資料分析初學者，正在 Google Colab 使用 pandas、numpy、matplotlib 或 scikit-learn。
我的任務是：刻意利用考後 quiz_score，依低於 75 產生 leaked_answer，比較與 actual；這是錯誤情境示範，不是可部署模型。
請依我貼上的上下文，只協助補上 ____，保留指定變數名稱與後續程式，不修改原始 df。
請解釋每一步的作用，提供繁體中文註解及可手動驗證的方法。
不要虛構執行結果；如需額外資訊，先指出缺少什麼。
```
完成後記錄實際使用的提示詞與自己改動的地方。


In [4]:
# 刻意錯誤示例：事後分數直接提供了標籤資訊
actual = (df["quiz_score"] < 75).astype(int)
leaked_answer = (df["quiz_score"] < 75).astype(int)
print("偷看答案的表面正確率：", (actual == leaked_answer).mean())
# 這個 100% 沒有實務預測效益：考前尚未取得 quiz_score


偷看答案的表面正確率： 1.0


### ✅ 作業 10-P1｜執行驗證
執行下方程式，並同時檢查上方實際輸出。自動檢查只涵蓋主要條件；如果圖表、說明或資料來源不符任務，仍需人工修正。


In [5]:
assert (leaked_answer == actual).all()
print("檢查通過：事後欄位可直接重現標籤；這正是洩漏警訊。")


檢查通過：事後欄位可直接重現標籤；這正是洩漏警訊。


### ✍️ 作業 10-P1｜我的完成紀錄
- 我實際使用的提示詞（若未使用 AI 請註明）：使用上方提供的 Vibe Coding 提示詞，詢問如何刻意利用考後 `quiz_score` 產生 `leaked_answer` 來示範目標洩漏。
- 我補上的程式在做什麼：輸入考後成績 `df["quiz_score"]`，透過 `< 75` 判斷並以 `.astype(int)` 轉成 `1`／`0` 存入 `leaked_answer`，刻意重現與定義目標標籤 `actual` 完全相同的規則。
- 執行或驗證遇到的問題與修正：沒有報錯，補完後直接執行 `10-P1` 與驗證 cell 皆順利通過。
- 檢查結果及我如何確認：表面正確率印出 `1.0`（100%），驗證 cell 也印出「檢查通過：事後欄位可直接重現標籤；這正是洩漏警訊。」。


## 10.4 Explain-back
請學生用自己的話區分：建立訓練標籤可用已知結果，但模型輸入只能用預測時間點可取得的資料。列出兩個真實專案會追問的欄位時間資訊。

### ✍️ 10.4 Explain-back 作答
- **用自己的話區分訓練標籤與模型輸入：**在訓練歷史資料時，我們是站在「事後回顧」的角度，所以必須用到已知的最終結果（例如考後的 `quiz_score` 或下個月是否退訂）來幫每筆資料貼上正確答案（標籤 `y`）；但是餵進模型的輸入特徵（`X`），則必須嚴格模擬「預測當下」的時間點，只能放進在那個時間點以前就已經發生且系統能查得到的資料，絕對不能把考後或事後才產生的資訊混進特徵裡。
- **真實專案會追問的兩個欄位時間資訊：**
  1. **統計截止時間（Cutoff Timestamp）：**這個彙總欄位（例如平均讀書時數或出席率）的計算區間是截止到「考試前一天 23:59」，還是不小心把考試當天甚至考後的紀錄也算進去了？
  2. **資料寫入與覆寫時間（Ingestion / Update Time）：**資料庫裡的欄位是考前就已經凍結留存的快照，還是系統會在期末或事後用最新狀態直接覆寫舊欄位（導致歷史資料表混入未來資訊）？


## 延伸：把時間點寫成資料規格
例如：以 1 月 31 日 23:59 為截點，預測 2 月是否退訂；登入次數的統計時間必須截止於 1 月 31 日。不能僅看欄位名稱就認定一定安全，也要檢查欄位的計算方式。

**學生練習：**為出席率、睡眠時間各寫下蒐集窗口、資料可得時間與一個可能洩漏的情境。

### ✍️ 延伸練習作答
- **出席率（`attendance_rate`）：**
  - **蒐集窗口：**開學第 1 週至小考前一次上課結束（截止於小考前一天 23:59）。
  - **資料可得時間：**每次上課點名結束後 24 小時內完成系統登錄，於小考前一天 23:59 前可由教務系統查詢。
  - **可能洩漏的情境：**若直接從學期末才匯出的總成績單取得 `attendance_rate`，該欄位會包含小考之後後半學期的出席紀錄，甚至受學生考差後是否缺課影響，造成未來資訊洩漏。
- **睡眠時間（`sleep_hours`）：**
  - **蒐集窗口：**小考前 7 天（含考前前一晚）的每日平均睡眠時數（截止於小考當天早上 08:00 前）。
  - **資料可得時間：**需在考前由穿戴裝置自動同步或考前問卷填答完畢並寫入資料庫。
  - **可能洩漏的情境：**若問卷是在「小考結束後」才發給學生回填這週平均睡幾小時，學生可能因為考不好而主觀低估自己的睡眠時間（事後回憶偏誤），導致考後狀態反向影響特徵值。


## 本單元學習收束與 AI 使用紀錄

請學生先用自己的話解釋一個關鍵觀念，再指出一個執行結果及驗證方式。記錄：使用的提示詞、AI 回答中檢查或修改的地方、遇到的問題、修正後的證據與本資料限制。

**通用提示詞**：我是 Python 初學者，請逐步協助我完成上面的任務；每段程式加繁體中文註解，說明輸入、處理與輸出，提供兩項可實際執行的驗證，保留原始資料，不要虛構執行結果。

### ✍️ 本單元收束與 AI 使用紀錄作答
- **關鍵觀念解釋：**建立預測模型時必須嚴格區分「訓練標籤（可用已知的最終結果定義）」與「模型輸入特徵（只能使用預測截點前已產生的資訊）」；如果把事後才知道的欄位（例如考後成績 `quiz_score` 或退訂後的 `cancellation_date`）放進輸入特徵，就會發生目標洩漏，表面準確率再高在真實預測時也完全無法使用。
- **執行結果及驗證方式：**在 `10-P1` 刻意使用考後 `quiz_score < 75` 產生 `leaked_answer`，得出表面正確率 `1.0`（100%），並透過 `assert (leaked_answer == actual).all()` 驗證它只是重複標籤規則的洩漏警訊；在 10.1 與 10.2 節則分別透過 `availability` 與 `churn` 表格核對可用與應排除的事後欄位。
- **使用的提示詞：**使用單元內提供的 `10-Q1`、`10-Q2` 查詢提示詞以及 `10-P1` 的 Vibe Coding 提示詞。
- **AI 回答中檢查或修改的地方：**將 AI 說明目標洩漏與時間截點的通用原則，改寫成直接對應本資料集 `quiz_score`、10.2 節 `churn` 表格日期（`2026-02-05`、`2026-02-12`），以及延伸練習中 `attendance_rate` 與 `sleep_hours` 的具體規格。
- **遇到的問題與修正後的證據：**無報錯；補完 `10-P1` 後執行驗證 cell 印出「檢查通過：事後欄位可直接重現標籤；這正是洩漏警訊。」即確認示範正確。
- **本資料限制：**本資料集為公式生成的 60 筆模擬資料，表格內並無實際資料收集時間戳記，因此 `study_hours`、`sleep_hours`、`screen_time_hours` 與 `attendance_rate` 皆在考前取得僅為教學假設；且兩筆睡眠缺失與 15.0 小時異常值均在計分後人為加入。
